# Extracting book descriptions from Google Books API
<hr>

The purpose of this notebook is to use the ISBN-13 codes from our main dataset from Kaggle - Goodreads <strong>"book.csv"</strong> to perform a series of API calls to retrieve the corresponding book descriptions from Google.

We will save the new dataset into a csv so we can later in another notebook transform and join  multiple datasets to create the final dataset for production.

In [71]:
# Dependencies
import pandas as pd
import requests
import json

# Google API Key
from api_keys import gkey

In [87]:
# Load CSV. This is the dataset where we will get our ISBN-13 codes from.
csv_kaggle = 'Resources/goodreads_books.csv'

goodreads_df = pd.read_csv(csv_kaggle)
goodreads_df.head(2)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,NaN
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,NaN


## Cleaning
<hr>

### Remove bad rows

An extra column in the end was added manually because the dataset was not cleaned to the extent that every row had the same number of columns. There were extra commas (",") in the <strong>"goodreads_books.csv"</strong> file which resulted in some single columns being recognized two columns thus adding unwanted information towards the right of the dataframe.

We decided to resolve this by deleting those rows.

In [88]:
# Fill all the None values as integer 0
goodreads_df = goodreads_df.fillna(0)
goodreads_df.head(2)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher,Unnamed: 12
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.,0
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.,0


In [89]:
# Filter out bad rows with an extra column of information and drop that column
filt_wanted = goodreads_df.iloc[:,-1] == 0
goodreads_df = goodreads_df.loc[filt_wanted]
goodreads_df = goodreads_df.drop(columns=['Unnamed: 12'])
goodreads_df.head(2)

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.


### Remove duplicated titles

There are books with different ISBN-13 codes yet identical titles, for example, "The Richest Man in Babylon" with the ISBN-13 of  9780451205360 has 194 pages but the other one with the ISBN-13 of 9781419349997 only has 4 pages. This could be an instance where the duplicate is a variation of some sort of the original text. For the purpose of this project, we do not need them thus we will remove those rows.

In [90]:
# Drop rows with duplicated titles
goodreads_df.drop_duplicates(subset='title', keep=False, inplace=True)

In [91]:
# Reset index
goodreads_df.reset_index(drop=True, inplace=True)

In [ ]:
google_books_api_df = goodreads_df[['isbn13', 'title']].copy()
google_books_api_df.head(2)

In [ ]:
google_books_api_df['Google_Books_description'] = ''
google_books_api_df['retail_price'] = ''
google_books_api_df['currency'] = ''
google_books_api_df.head(2)

In [ ]:
# Use iterrows to iterate through pandas dataframe
for index, row in df[0:5].iterrows():

    # get restaurant type from df
    isbn_13 = row['isbn13']

    # Assemble url and make API request
    target_url = f'https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn_13}&key={gkey}'
    
    print(f"Retrieving Results for ISBN-13: {isbn_13}...")
    response = requests.get(target_url).json()
    
    try:
        
        # Extract results
        title = response['items'][0]['volumeInfo']['title']
        description = response['items'][0]['volumeInfo']['description']

        print(f'Description for Volume: {title}, ISBN-13: {isbn_13} retrieved...')
        
        df.loc[index, 'Google_Books_description'] = description
        
    except (KeyError, IndexError):
        print("Missing field/result... skipping.")
        
    print("------------")


In [ ]:
df.head(5)

In [ ]:
target_url = f'https://www.googleapis.com/books/v1/volumes?q=isbn:9780439785969&key={gkey}'
rating_data = requests.get(target_url).json()
rating_data

In [ ]:
description = rating_data['items'][0]['volumeInfo']['description']

In [ ]:
description